# Tier 2 Assignment 1 — Solutions: Sequences and Alignment

Complete solutions for all 8 problems.


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIXML
import io

## Problem 1: Database Retrieval

Use Entrez to fetch a GenBank record and extract organism, description, length,
and the first 100 nucleotides.

In [ ]:
def fetch_genbank_record(accession: str, email: str) -> dict:
    Entrez.email = email
    with Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text") as handle:
        record = SeqIO.read(handle, "genbank")
    return {
        "organism": record.annotations.get("organism", "unknown"),
        "description": record.description,
        "length": len(record.seq),
        "first_100_nt": str(record.seq[:100]),
    }


# Demo (requires network access)
# result = fetch_genbank_record("NM_000207", "your.email@example.com")
# for key, value in result.items():
#     print(f"{key}: {value}")

# Expected output:
# organism: Homo sapiens
# description: Homo sapiens insulin (INS), mRNA
# length: 465
# first_100_nt: AGCCCTCCAGGACAGGCTGCAT...
print("fetch_genbank_record defined. Uncomment demo lines to run (requires network).")

## Problem 2: FASTA Statistics

Parse a FASTA file with BioPython SeqIO and compute summary statistics.

In [ ]:
def fasta_statistics(fasta_path: str) -> dict:
    records = list(SeqIO.parse(fasta_path, "fasta"))
    if not records:
        return {"num_sequences": 0, "avg_length": 0.0, "min_length": 0, "max_length": 0, "avg_gc": 0.0}

    lengths = [len(r.seq) for r in records]

    def gc_fraction(seq: str) -> float:
        seq = seq.upper()
        return (seq.count('G') + seq.count('C')) / len(seq) * 100 if seq else 0.0

    gc_values = [gc_fraction(str(r.seq)) for r in records]

    return {
        "num_sequences": len(records),
        "avg_length": round(sum(lengths) / len(lengths), 2),
        "min_length": min(lengths),
        "max_length": max(lengths),
        "avg_gc": round(sum(gc_values) / len(gc_values), 2),
    }


# Test
import tempfile, os

sample_fasta = """>seq1 Human insulin
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTGG
>seq2 Mouse insulin
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTGG
ATGAGCCTCCAGGACAGGCTGCATCAGAAGAGGCC
>seq3 Rat insulin
ATGAGCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACT
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as f:
    f.write(sample_fasta)
    tmp_path = f.name

stats = fasta_statistics(tmp_path)
for key, value in stats.items():
    print(f"{key}: {value}")

os.unlink(tmp_path)

## Problem 3: Dot Plot

Build an n×m boolean match matrix then print it with column/row headers.
The window trick: only mark cell (i,j) if the subsequence of length `window`
starting at each position is identical.

In [ ]:
def dot_plot(seq1: str, seq2: str, window: int = 1) -> None:
    n, m = len(seq1), len(seq2)
    # Build match matrix; positions within `window` bp of the end are False
    matrix = [
        [
            seq1[i:i + window] == seq2[j:j + window]
            if i + window <= n and j + window <= m
            else False
            for j in range(m)
        ]
        for i in range(n)
    ]

    # Column header
    col_header = "    " + " ".join(seq2)
    print(col_header)
    for i, row in enumerate(matrix):
        cells = " ".join("." if cell else " " for cell in row)
        print(f"  {seq1[i]} {cells}")


# Test
s1 = "ATGCATGC"
s2 = "ATGCTTGC"
print("Window=1:")
dot_plot(s1, s2, window=1)
print("\nWindow=3:")
dot_plot(s1, s2, window=3)

## Problem 4: Pairwise Alignment Scoring

Walk column-by-column. A gap is any position where either character is `'-'`;
a match is two identical non-gap characters; a mismatch is two different non-gap
characters. Identity = matches / (non-gap columns). Similarity uses the same
denominator but counts conserved-group substitutions as matches.

In [ ]:
# Conserved substitution groups (BLOSUM-inspired, for protein)
_SIMILAR_GROUPS = [
    set("ILVM"), set("FYW"), set("KRH"), set("DE"), set("STNQ"), set("AG"), set("CP")
]


def _is_similar(a: str, b: str) -> bool:
    return any(a in g and b in g for g in _SIMILAR_GROUPS)


def score_alignment(aligned_seq1: str, aligned_seq2: str) -> dict:
    if len(aligned_seq1) != len(aligned_seq2):
        raise ValueError("Aligned sequences must have equal length.")

    matches = mismatches = gaps = 0
    similar = 0

    for a, b in zip(aligned_seq1, aligned_seq2):
        if a == '-' or b == '-':
            gaps += 1
        elif a == b:
            matches += 1
            similar += 1
        else:
            mismatches += 1
            if _is_similar(a, b):
                similar += 1

    aligned_cols = matches + mismatches  # non-gap columns
    total_cols = len(aligned_seq1)
    identity = (matches / total_cols * 100) if total_cols else 0.0
    similarity = (similar / total_cols * 100) if total_cols else 0.0

    return {
        "matches": matches,
        "mismatches": mismatches,
        "gaps": gaps,
        "percent_identity": round(identity, 2),
        "percent_similarity": round(similarity, 2),
    }


# Test
a1 = "ATCG-ATCG"
a2 = "ATCGATC-G"
result = score_alignment(a1, a2)
print(f"{a1}\n{a2}")
for key, value in result.items():
    print(f"{key}: {value}")

## Problem 5: BLAST Parser

`NCBIXML.parse` returns an iterator of `Blast` objects. Each has `.alignments`,
each alignment has `.title`, `.accession`, and `.hsps`. We take the first HSP.

In [ ]:
def parse_blast_xml(xml_source) -> list[dict]:
    if isinstance(xml_source, str):
        handle = open(xml_source)
        close_after = True
    else:
        handle = xml_source
        close_after = False

    hits = []
    for record in NCBIXML.parse(handle):
        for alignment in record.alignments:
            hsp = alignment.hsps[0]  # first HSP
            e_value = hsp.expect
            if e_value >= 0.001:
                continue
            pct_identity = hsp.identities / hsp.align_length * 100
            hits.append({
                "accession": alignment.accession,
                "description": alignment.title,
                "e_value": e_value,
                "percent_identity": round(pct_identity, 2),
                "alignment_length": hsp.align_length,
            })

    if close_after:
        handle.close()

    return sorted(hits, key=lambda h: h["e_value"])


# Test with the sample XML from the assignment
BLAST_XML_SAMPLE = """<?xml version="1.0"?>
<!DOCTYPE BlastOutput PUBLIC "-//NCBI//NCBI BlastOutput/EN"
    "http://www.ncbi.nlm.nih.gov/dtd/NCBI_BlastOutput.dtd">
<BlastOutput>
  <BlastOutput_program>blastn</BlastOutput_program>
  <BlastOutput_iterations>
    <Iteration>
      <Iteration_hits>
        <Hit>
          <Hit_accession>NM_000207</Hit_accession>
          <Hit_def>Homo sapiens insulin (INS), mRNA</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>1e-50</Hsp_evalue>
              <Hsp_identity>98</Hsp_identity>
              <Hsp_align-len>100</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
        <Hit>
          <Hit_accession>NM_001185097</Hit_accession>
          <Hit_def>Mus musculus insulin (Ins2), mRNA</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>1e-30</Hsp_evalue>
              <Hsp_identity>85</Hsp_identity>
              <Hsp_align-len>100</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
        <Hit>
          <Hit_accession>XM_999999</Hit_accession>
          <Hit_def>Weak hit that should be filtered out</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>0.5</Hsp_evalue>
              <Hsp_identity>40</Hsp_identity>
              <Hsp_align-len>80</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
      </Iteration_hits>
    </Iteration>
  </BlastOutput_iterations>
</BlastOutput>
"""

hits = parse_blast_xml(io.StringIO(BLAST_XML_SAMPLE))
print(f"Hits passing e-value filter: {len(hits)}")
for hit in hits:
    print(f"  {hit['accession']}: e={hit['e_value']}, identity={hit['percent_identity']}%")

## Problem 6: Multiple Alignment Conservation

For each column count non-gap characters; the score is the fraction of the
most common one. All-gap columns score 0.

In [ ]:
from collections import Counter


def conservation_scores(alignment: list[str]) -> list[float]:
    if not alignment:
        return []
    width = len(alignment[0])
    scores = []
    for col in range(width):
        chars = [seq[col] for seq in alignment if seq[col] != '-']
        if not chars:
            scores.append(0.0)
        else:
            most_common_count = Counter(chars).most_common(1)[0][1]
            scores.append(most_common_count / len(chars))
    return scores


def top_conserved_positions(alignment: list[str], n: int = 10) -> list[tuple]:
    scores = conservation_scores(alignment)
    indexed = sorted(enumerate(scores, start=1), key=lambda x: x[1], reverse=True)
    return [(pos, round(score, 4)) for pos, score in indexed[:n]]


# Test
test_msa = [
    "ACGT-ACGT",
    "ACGT-ACGG",
    "ACGT-ACGA",
    "ACGG-ACGT",
    "ACGT-ACGT",
]
scores = conservation_scores(test_msa)
print("Conservation scores:", [round(s, 3) for s in scores])
print("Top 5 positions:", top_conserved_positions(test_msa, n=5))

## Problem 7: Distance Matrix

p-distance = (number of differing non-gap sites) / (number of non-gap sites).
Print with fixed-width column headers for readability.

In [ ]:
def p_distance(seq1: str, seq2: str) -> float:
    diffs = 0
    compared = 0
    for a, b in zip(seq1, seq2):
        if a == '-' or b == '-':
            continue
        compared += 1
        if a != b:
            diffs += 1
    return diffs / compared if compared else 0.0


def distance_matrix(sequences: dict[str, str]) -> None:
    names = list(sequences.keys())
    seqs = list(sequences.values())
    n = len(names)

    # Compute full matrix
    matrix = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = p_distance(seqs[i], seqs[j])
            matrix[i][j] = d
            matrix[j][i] = d

    # Print formatted table
    col_w = max(len(name) for name in names) + 2
    header = " " * col_w + "".join(f"{name:>{col_w}}" for name in names)
    print(header)
    for i, name in enumerate(names):
        row = f"{name:<{col_w}}" + "".join(f"{matrix[i][j]:>{col_w}.3f}" for j in range(n))
        print(row)


# Test
test_seqs = {
    "Human":     "ATGCATGCATGCATGC",
    "Mouse":     "ATGCATGCATGCATGA",
    "Rat":       "ATGCATGCATGCATGG",
    "Zebrafish": "ATGCATGCTTGCATGC",
}
distance_matrix(test_seqs)

## Problem 8: Simple UPGMA

Maintain a list of cluster labels (Newick sub-trees), a distance matrix, and
cluster sizes. At each step: find the minimum off-diagonal entry, merge the two
clusters, update distances using the weighted average, and remove the old rows/cols.

In [ ]:
def upgma(names: list[str], dist: list[list[float]]) -> str:
    import copy

    labels = list(names)
    matrix = [list(row) for row in dist]
    sizes = [1] * len(labels)

    while len(labels) > 1:
        n = len(labels)

        # Find the pair with minimum distance
        min_dist = float('inf')
        min_i, min_j = 0, 1
        for i in range(n):
            for j in range(i + 1, n):
                if matrix[i][j] < min_dist:
                    min_dist = matrix[i][j]
                    min_i, min_j = i, j

        # Branch length to each child is half the distance between them
        branch = min_dist / 2
        new_label = f"({labels[min_i]}:{branch:.4f},{labels[min_j]}:{branch:.4f})"
        new_size = sizes[min_i] + sizes[min_j]

        # Compute new distances (weighted average)
        new_row = []
        for k in range(n):
            if k == min_i or k == min_j:
                continue
            d = (sizes[min_i] * matrix[min_i][k] + sizes[min_j] * matrix[min_j][k]) / new_size
            new_row.append(d)

        # Remove the two merged clusters (higher index first to preserve indices)
        for idx in sorted([min_i, min_j], reverse=True):
            labels.pop(idx)
            sizes.pop(idx)
            matrix.pop(idx)
            for row in matrix:
                row.pop(idx)

        # Add new cluster
        labels.append(new_label)
        sizes.append(new_size)
        for i, row in enumerate(matrix):
            row.append(new_row[i])
        matrix.append(new_row + [0.0])

    return labels[0] + ";"


# Test — symmetric 4-taxon example; expected grouping: (A,B) and (C,D)
taxa = ["A", "B", "C", "D"]
distances = [
    [0.0, 0.2, 0.4, 0.4],
    [0.2, 0.0, 0.4, 0.4],
    [0.4, 0.4, 0.0, 0.2],
    [0.4, 0.4, 0.2, 0.0],
]
tree = upgma(taxa, distances)
print("Newick tree:")
print(tree)
# Expected: ((A:0.1000,B:0.1000):0.1000,(C:0.1000,D:0.1000):0.1000);

---

## Key Patterns

1. **Entrez**: always set `Entrez.email`; use context manager for the handle
2. **SeqIO.parse**: returns an iterator; wrap in `list()` when you need random access
3. **Dot plot**: build the full matrix first, then render — clean separation
4. **Alignment scoring**: walk column-by-column with `zip`
5. **BLAST XML**: `NCBIXML.parse` handles multi-query files; `hsp.expect` is the e-value
6. **Conservation**: `Counter.most_common(1)[0][1]` to get the max count
7. **p-distance**: skip gap positions explicitly before dividing
8. **UPGMA**: always remove higher index first when deleting two entries from a list
